# Sticky Sampler Validation

For each spike-and-slab target we run the sticky Boomerang (exact and PLI),
resample uniformly in time, and check:

1. **Marginals**: histograms against the true spike-and-slab density $w_i\,\delta_0 + (1-w_i)\,\text{slab}_i$.
2. **Frozen fractions**: empirical $\hat{w}_i$ (fraction of time coordinate $i$ is at zero) vs the target spike weight $w_i$.
3. **Inclusion probabilities** (linear regression): sampler inclusion vs exact enumeration.

In [ ]:
import os
os.chdir('../..')

import numpy as np
import matplotlib.pyplot as plt

from benchmarks_august.targets.validation_sticky import spike_slab_gaussian, spike_slab_linreg, spike_slab_logreg
from benchmarks_august.samplers.factories import build_sampler
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from benchmarks_august.samplers.warmstart import warmup_reference

In [ ]:
# ── Shared settings ──────────────────────────────────────────────
N_SKELETON  = 20000
N_RESAMPLE  = 50000
BURNIN_FRAC = 0.1
refresh_rate = 1.0
ZERO_TOL = 1e-8  # threshold for counting a sample as "at zero"

def run_and_resample(sampler, target, sticky=True, warmup=True, N_SKELETONS=N_SKELETON):
    """Warmup, preprocess, sample, and return time-uniform resamples."""
    if warmup:
        warmup_reference(sampler, n_rounds=3, n_pilot=500,
                         sticky=sticky, target=target)
    else:
        method = target.meta.get('preprocess_method', 'diagonal')
        if method == 'manual':
            sampler.preprocess(method='manual',
                               x_ref=target.x_ref,
                               Sigma_inv=target.Sigma_inv)
        else:
            sampler.preprocess(method='diagonal')
    
    sampler.reset(N=N_SKELETONS)
    sampler.sample_auto(diagnostics=True)
    
    if sticky:
        _, samples = resample_sticky_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                               burnin_frac=BURNIN_FRAC)
    else:
        _, samples = resample_pdmp_path(sampler, n_samples=N_RESAMPLE,
                                        burnin_frac=BURNIN_FRAC)
    return samples


def frozen_fractions(samples, tol=ZERO_TOL):
    """Fraction of resampled points where each coordinate is exactly zero."""
    return np.mean(np.abs(samples) < tol, axis=0)


def plot_marginals_sticky(target, samples_dict, figname=None, bins=80):
    """Plot marginal histograms against the TRUE spike-and-slab density.
    
    The true marginal is  w_i * delta_0 + (1 - w_i) * slab_i(beta).
    We plot (1 - w_i) * slab as a curve, and annotate the expected
    spike mass w_i. The histogram should show a spike at zero whose
    height (in the zero-bin) matches the spike weight.
    """
    marginals = target.meta['marginal_grids']
    D = target.D
    n_samplers = len(samples_dict)

    fig, axes = plt.subplots(n_samplers, D, figsize=(3.5 * D, 3 * n_samplers),
                             squeeze=False)

    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']

    for row, (label, samples) in enumerate(samples_dict.items()):
        for col in range(D):
            ax = axes[row, col]
            mg = marginals[col]
            grid, pdf = mg['grid'], mg['pdf']
            w_i = mg['spike_weight']

            x = samples[:, col]
            x_lo, x_hi = float(grid[0]), float(grid[-1])

            # Separate zero (frozen) and nonzero (active) samples
            is_zero = np.abs(x) < ZERO_TOL
            x_active = x[~is_zero]
            frac_zero = is_zero.mean()

            # Plot histogram of active (nonzero) samples, scaled by (1 - frac_zero)
            if len(x_active) > 0:
                weights_active = np.ones_like(x_active) * (1 - frac_zero) / len(x_active)
                bin_width = (x_hi - x_lo) / bins
                weights_active /= bin_width
                ax.hist(x_active, bins=bins, range=(x_lo, x_hi),
                        weights=weights_active,
                        alpha=0.5, color=colors[row % len(colors)])

            # Truth: (1 - w_i) * slab density
            ax.plot(grid, (1 - w_i) * pdf, 'k-', lw=1.5)

            # Annotate spike mass
            ax.text(0.98, 0.95,
                    f'$\\hat{{w}}$={frac_zero:.2f}\ntrue={w_i:.2f}',
                    transform=ax.transAxes, fontsize=7,
                    va='top', ha='right',
                    color='green' if abs(frac_zero - w_i) < 0.1 else 'red')

            ax.set_xlim(x_lo, x_hi)
            if row == 0:
                ax.set_title(mg['label'])
            if col == 0:
                ax.set_ylabel(label)

    fig.suptitle(target.name, fontsize=14, y=1.02)
    fig.tight_layout()
    if figname:
        fig.savefig(figname, dpi=150, bbox_inches='tight')
    plt.show()


def plot_frozen_fractions(target, samples_dict):
    """Bar chart comparing empirical frozen fractions to target spike weights."""
    w_true = target.meta['spike_weights']
    D = target.D
    x_pos = np.arange(D)
    width = 0.8 / (len(samples_dict) + 1)

    fig, ax = plt.subplots(figsize=(max(6, D * 0.8), 4))
    ax.bar(x_pos - 0.4 + width * 0.5, w_true, width, label='Target $w_i$',
           color='black', alpha=0.3)

    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']
    for j, (label, samples) in enumerate(samples_dict.items()):
        ff = frozen_fractions(samples)
        offset = -0.4 + width * (j + 1.5)
        ax.bar(x_pos + offset, ff, width, label=label,
               color=colors[j % len(colors)], alpha=0.7)

    ax.set_xlabel('Coordinate')
    ax.set_ylabel('Frozen fraction')
    ax.set_title(f'{target.name}: frozen fractions vs target spike weights')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'$\\beta_{{{i+1}}}$' for i in range(D)])
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    plt.show()


def plot_inclusion_probs(target, samples_dict):
    """Compare sampler inclusion probabilities to exact values (linreg only)."""
    if 'inclusion_probs' not in target.meta:
        print('No exact inclusion probabilities available for this target.')
        return

    inc_exact = target.meta['inclusion_probs']
    D = target.D

    fig, ax = plt.subplots(figsize=(max(6, D * 0.8), 4))
    x_pos = np.arange(D)

    ax.plot(x_pos, inc_exact, 'ks', ms=10, label='Exact $P(\\beta_i \\neq 0 \\mid y)$', zorder=5)

    colors = ['steelblue', 'darkorange', 'seagreen', 'firebrick']
    markers = ['o', '^', 'D', 'v']
    for j, (label, samples) in enumerate(samples_dict.items()):
        ff = frozen_fractions(samples)
        inc_sampler = 1.0 - ff  # inclusion = 1 - frozen fraction
        ax.plot(x_pos, inc_sampler, markers[j % len(markers)],
                color=colors[j % len(colors)], ms=8, label=label)

    # Annotate true beta values
    if target.true_params is not None:
        for i in range(D):
            ax.text(i, -0.08, f'{target.true_params[i]:.1f}',
                    ha='center', fontsize=7, color='gray')
        ax.text(D / 2, -0.15, r'(true $\beta$)', ha='center',
                fontsize=7, color='gray')

    ax.set_xlabel('Coordinate')
    ax.set_ylabel('Inclusion probability')
    ax.set_title(f'{target.name}: inclusion probabilities')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'$\\beta_{{{i+1}}}$' for i in range(D)])
    ax.set_ylim(-0.2, 1.1)
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.axhline(1, color='gray', lw=0.5, ls='--')
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 1. Gaussian sanity check

With a Gaussian slab and known $\Sigma$, the reference measure cancels the potential exactly.
There should be **zero accepted bounces** — all skeleton points come from refreshments.
The frozen fraction of coordinate $i$ should match the spike weight $w_i$.

In [ ]:
diag_gauss = spike_slab_gaussian(D=5)
ar_gauss = spike_slab_gaussian(D=5, cov="ar1")
random_gauss = spike_slab_gaussian(D=5, cov="random")

target_gauss = diag_gauss
kappa_manual = 0.5*np.ones(5)#np.array([0.5, 1.0, 2.0, 4.0, 8.0])

s_gauss = build_sampler('sticky_boomerang', target_gauss, N=N_SKELETON,
                        refresh_rate=refresh_rate, kappa=kappa_manual)
samp_gauss = run_and_resample(s_gauss, target_gauss, warmup=False)

s_gauss_pli = build_sampler('sticky_boomerang_pli', target_gauss, N=N_SKELETON,
                            refresh_rate=refresh_rate, kappa=kappa_manual)
samp_gauss_pli = run_and_resample(s_gauss_pli, target_gauss, warmup=False)

In [ ]:
# Sanity: should be zero bounces for Gaussian target
for name, s in [('Sticky Boomerang', s_gauss), ('Sticky Boomerang PLI', s_gauss_pli)]:
    df = s.diagnostics_df
    n_bounces = df[(df['event_type'] == 'bounce') & (df['accepted'] == True)].shape[0]
    n_refresh = df[df['event_type'] == 'refresh'].shape[0]
    print(f'--- {name} ---')
    print(f'  Bounces: {n_bounces}  (expect 0)')
    print(f'  Refreshments: {n_refresh}')
    print(f'  Grad evals/skeleton: {s.gradient_evals / s.N:.1f}')
    print(f'  Wall time: {df["wall_seconds"].sum():.2f}s')

In [ ]:
gauss_samples = {'Sticky Boomerang': samp_gauss,
                 'Sticky Boomerang PLI': samp_gauss_pli}

plot_marginals_sticky(target_gauss, gauss_samples)
plot_frozen_fractions(target_gauss, gauss_samples)

## 2. Linear regression (conjugate, exact inclusion probabilities)

The key validation here is the **inclusion probability plot**: the sampler's
$1 - \hat{w}_i$ should match the exact $P(\beta_i \neq 0 \mid y)$ computed
by enumerating all $2^p$ models.

In [ ]:
target_regression = spike_slab_linreg(n=100, p=8, sparsity="sparse")

print('Exact inclusion probabilities:')
for i, p_inc in enumerate(target_regression.meta['inclusion_probs']):
    true_beta = target_regression.true_params[i]
    print(f'  beta_{i+1}: P(incl|y) = {p_inc:.4f}   (true = {true_beta:.1f})')

In [ ]:
target_regression.meta['kappa']

In [ ]:
s_linreg = build_sampler('sticky_boomerang', target_regression, N=N_SKELETON,
                         refresh_rate=0.1, kappa=target_regression.meta['kappa'])
samp_linreg = run_and_resample(s_linreg, target_regression, warmup=False)

In [ ]:
s_linreg_pli = build_sampler('sticky_boomerang_pli', target_regression, N=N_SKELETON,
                              refresh_rate=0.1, kappa=target_regression.meta['kappa'])
samp_linreg_pli = run_and_resample(s_linreg_pli, target_regression, warmup=False)

In [ ]:
linreg_samples = {'Sticky Boomerang': samp_linreg,
                  'Sticky Boomerang PLI': samp_linreg_pli}

plot_marginals_sticky(target_regression, linreg_samples)
plot_frozen_fractions(target_regression, linreg_samples)
plot_inclusion_probs(target_regression, linreg_samples)

## 3. Logistic regression (no closed-form inclusion probabilities)

Validation is qualitative: coordinates with $\beta^\text{true}_i = 0$ should
have **high** frozen fractions, and active coordinates should have **low**
frozen fractions. The signal should sharpen with increasing $n$.

In [ ]:
target_logreg = spike_slab_logreg(n=100, p=8, sparsity="sparse")

print('True beta:', target_logreg.true_params)
print('Expected high frozen fraction for coords:', 
      np.where(target_logreg.true_params == 0)[0] + 1)

In [ ]:
s_logreg = build_sampler('sticky_boomerang', target_logreg, N=N_SKELETON,
                         refresh_rate=0.1, kappa=target_logreg.meta['kappa'])
samp_logreg = run_and_resample(s_logreg, target_logreg, warmup=False)

In [ ]:
s_logreg_pli = build_sampler('sticky_boomerang_pli', target_logreg, N=N_SKELETON,
                              refresh_rate=0.1, kappa=target_logreg.meta['kappa'])
samp_logreg_pli = run_and_resample(s_logreg_pli, target_logreg, warmup=False)

In [ ]:
logreg_samples = {'Sticky Boomerang': samp_logreg,
                  'Sticky Boomerang PLI': samp_logreg_pli}

plot_marginals_sticky(target_logreg, logreg_samples)
plot_frozen_fractions(target_logreg, logreg_samples)

## 4. Posterior contraction check (logistic regression)

Increase $n$ and verify that inclusion probabilities concentrate:
active coords $\to 1$, null coords $\to 0$.

In [ ]:
n_values = [50, 200, 500]
p = 8

fig, axes = plt.subplots(1, len(n_values), figsize=(5 * len(n_values), 4),
                         sharey=True)

for ax, n_obs in zip(axes, n_values):
    t = spike_slab_logreg(n=n_obs, p=p, sparsity='sparse')
    s = build_sampler('sticky_boomerang_pli', t, N=N_SKELETON,
                      refresh_rate=0.1, kappa=t.meta['kappa'])
    samp = run_and_resample(s, t, warmup=False)
    
    ff = frozen_fractions(samp)
    inc = 1.0 - ff
    
    x_pos = np.arange(p)
    is_active = t.true_params != 0
    colors_bar = ['steelblue' if a else 'lightcoral' for a in is_active]
    ax.bar(x_pos, inc, color=colors_bar, alpha=0.7)
    ax.set_title(f'n = {n_obs}')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'$\\beta_{{{i+1}}}$' for i in range(p)], fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.axhline(1, color='gray', lw=0.5, ls='--')
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    if ax == axes[0]:
        ax.set_ylabel('Inclusion probability')

fig.suptitle('Posterior contraction: blue = truly active, red = truly null', fontsize=11)
fig.tight_layout()
plt.show()